## 1. Question

**Can RAG explain a song recommendation using only the supplied catalog evidence, while acknowledging when the connection is weak?** The previous notebook returned ranked, unheard songs. This notebook adds explanations that readers can check against the liked and recommended songs’ records.

The explanations concern textual connections. They cannot establish whether the listener would enjoy a song or reveal the embedding model’s internal reasoning.

## 2. What Is RAG?

Consider two questions for ChatGPT. “Who was Alan Turing?” can often be answered from what the model learned during training. “What is the latest NASA announcement?” needs a current source: the announcement may have been published after the model’s knowledge cutoff date. With web search enabled, ChatGPT’s search tool retrieves recent NASA pages and adds relevant excerpts to the model’s prompt. The model uses those excerpts to answer.

Our recommendation question follows the same sequence. The model cannot explain the textual connection between *Last Letter* and the listener’s likes from training alone because our song catalog and listener history are fictional. We first find the liked songs and *Last Letter* in the catalog, then give their descriptions to the model. Now its explanation can refer to evidence we can check.

This sequence is **Retrieval-Augmented Generation (RAG)**:

- **Retrieval:** select relevant records from an external collection.
- **Augmentation:** place those records, source identifiers, and instructions into the model’s prompt.
- **Generation:** produce an answer using the supplied context.

The catalog records reach the model when it answers; its weights are not updated. Supplying evidence helps us ask for a supported explanation, but we still need to check the answer against those records.


### Example: explaining a recommendation

![A RAG pipeline retrieves the listener's liked songs and the recommended song, supplies their catalog records to a language model, and returns an evidence-based explanation.](rag-song-explanation.svg)

*Illustrative response, based on the catalog descriptions; the model has not generated this example.*

### The RAG pipeline in this song example

The diagram shows two paths through the RAG pipeline using our fictional songs. **Ingestion** prepares the catalog text for search. **Retrieval** finds song passages that can help answer a listener’s question. Those passages then become context for the language model.

![RAG pipeline diagram: split song descriptions and notes into chunks, embed and index them, retrieve relevant song chunks for a listener's question, then add them to a prompt for a language model.](rag-document-pipeline.svg)

#### Ingestion: make the song text searchable

- **Load and chunk.** Treat each title and description in `songs.csv` as a short song passage. One description can be one chunk; longer notes about a song can be split into several chunks. Keep the song ID with every chunk so an explanation can point back to its source.
- **Embed and index.** Use Ollama’s `mxbai-embed-large` to turn each chunk into a vector. Store the chunk text, song ID, and vector in Chroma for later search.

#### Retrieval: find evidence about the liked and recommended songs

- **Retrieve.** For “Why was *Last Letter* recommended to this listener?”, embed the question with the same model and search Chroma for relevant song chunks. Also include the known records for *First Light*, *Begin Again*, *Morning Resolve*, and *Last Letter* by song ID; similarity search alone might miss one of them.

#### Augmentation and generation: answer from that evidence

- **Augment.** Fill a prompt template with the listener’s question, the selected song chunks and their IDs, and instructions to explain shared themes and acknowledge differences.
- **Generate.** Send that prompt to a language model. It can connect *Last Letter*’s goodbye and letting-go theme with the liked songs’ breakup and recovery themes, while noting its bittersweet mood. We then check each claim against the supplied song text.

The previous notebook’s listener vector retrieves a candidate song. This RAG retrieval gathers textual evidence to explain that candidate; it does not reveal the embedding model’s internal reasoning.


## 4. Experiment Setup

We build on the previous notebook’s 24-song catalog and listener histories. Its centroid-based recommender used SBERT’s `all-MiniLM-L6-v2` to rank unheard songs, with Chroma storing and searching their embeddings. Here we add a separate text-retrieval path to gather evidence for each explanation.

### Embedding model for the RAG evidence

We used SBERT’s `all-MiniLM-L6-v2` in the earlier notebooks to demonstrate how vector databases search by meaning. For this RAG evidence, we’ll switch to Ollama’s `mxbai-embed-large`. Ollama [reports strong retrieval benchmark results](https://ollama.com/library/mxbai-embed-large) for this open-weight model, and it can run locally. That makes it a practical alternative to a commercial embedding API for this experiment.

We’ll use the same model to embed song chunks during ingestion and the listener’s question during retrieval.


### Get the model and Python libraries

Start Ollama, then download the embedding model in a terminal:

```sh
ollama pull mxbai-embed-large
```

From the repository root, create the project environment if needed and install the pinned dependencies:

```sh
python3.10 -m venv .venv
.venv/bin/python -m pip install -r requirements.txt
```

If you are installing only this example's libraries, the equivalent package command is:

```sh
.venv/bin/python -m pip install "fastapi[standard]==0.119.0" chromadb==1.2.1 \
  ipykernel==6.29.4 jinja2==3.1.6 langchain-text-splitters==1.0.0 \
  ollama==0.6.3 requests==2.32.5
```

The RAG example uses these libraries:

| Package | Purpose |
| --- | --- |
| `fastapi[standard]` | Serve the example API. |
| `ollama` | Call the local Ollama API from Python. |
| `chromadb` | Store and retrieve song chunk embeddings. |
| `jinja2` | Fill the explanation prompt template. |
| `langchain-text-splitters` | Split longer song text into manageable chunks. |
| `requests` | Make HTTP requests to the example API. |

`langchain-text-splitters` supplies the chunking utility; we do not need the full LangChain framework for this pipeline. Select `.venv/bin/python` as the notebook kernel when running the RAG code.


### Data ingestion: preparing song passages

![Song data ingestion pipeline: split song text into ID-linked chunks, embed each chunk, and store text, IDs, and vectors in Chroma.](rag-song-ingestion.svg)

Loading the vector database usually happens **before** anyone asks a question and it is generally an offline process. It may be a one-time load, or data may be continually added or updated by a background process. We might ingest the catalog once, then run a background job whenever song descriptions or notes change. This gives the retrieval step a searchable collection to use later.

For this example, we take the text in `songs.csv` and any longer song notes, split it into meaningful passages, and keep the song ID with every chunk. We embed each chunk with `mxbai-embed-large`, then store the chunk text, its song ID, and its vector in Chroma. Short descriptions can remain a single chunk; longer notes may need several.

**Why chunk at all?** When the listener asks why *Last Letter* was recommended, the system needs to retrieve the passage that describes its relevant themes before the LLM can explain the match. A chunk that mixes too many topics can be hard to match; one that is too small can lose the context needed for a useful answer. The same principle applies when answering a question about a particular passage in documentation: first make that passage findable.


### Understanding Chunking Basics

Choosing a good text-chunking strategy is key to building a reliable RAG solution. Several strategies are available, and the right choice balances three goals:

- **Capture meaning.** Accurately represent the source text in each embedding.
- **Include relevant context.** Keep enough contextually relevant information in each chunk.
- **Stay within token limits.** Keep each chunk within the embedding model's request limit.

We can experiment with different strategies for our specific use case. Chunking also provides practical benefits:

- Smaller chunks capture fewer topics or ideas.
- More focused embeddings can improve retrieval accuracy.
- Shorter chunks help avoid token limits when generating embeddings.
- Retrieved chunks send fewer, more relevant tokens to the LLM.

#### Start with `CharacterTextSplitter`

To put those chunking goals into practice, we’ll start with `CharacterTextSplitter` from LangChain’s `langchain-text-splitters` package. It is a simple splitter that uses a chosen separator. In this example, blank lines mark paragraph boundaries in an **illustrative** longer note about *Last Letter*. We use `create_documents` because it calls `split_text` and attaches the song ID and title to each resulting chunk. Calling `split_text` directly would return only the chunk strings.

Here, `chunk_size` is a target measured in characters, not tokens. The splitter keeps a paragraph intact if it is longer than that target, so we should still check chunk lengths before sending them to an embedding model.


In [6]:
from langchain_text_splitters import CharacterTextSplitter

# Use a longer illustrative note; the CSV description fits in one chunk.
song_note = (
    "Last Letter is a bittersweet folk song about saying goodbye to a former partner.\n\n"
    "Its narrator recalls the relationship while deciding to let go.\n\n"
    "The song ends with acceptance rather than a promise to reunite."
)

# Split at paragraph breaks with a 110-character target and no overlap.
# Paragraphs longer than the target stay intact.
splitter = CharacterTextSplitter(separator="\n\n", chunk_size=110, chunk_overlap=0)
# Keep the song ID and title on each chunk so we can trace its source.
song_chunks = splitter.create_documents(
    [song_note], metadatas=[{"song_id": "song_008", "title": "Last Letter"}]
)
for number, chunk in enumerate(song_chunks, start=1):
    print(f"Chunk {number} ({len(chunk.page_content)} characters): {chunk.page_content}")
    print(f"  source: {chunk.metadata['song_id']}\n")


Chunk 1 (80 characters): Last Letter is a bittersweet folk song about saying goodbye to a former partner.
  source: song_008

Chunk 2 (63 characters): Its narrator recalls the relationship while deciding to let go.
  source: song_008

Chunk 3 (63 characters): The song ends with acceptance rather than a promise to reunite.
  source: song_008



#### Overlap and character boundaries

Overlapping text between chunks can:

- Reduce the chance that an idea is cut between consecutive chunks.
- Give consecutive chunks shared context and greater semantic similarity.

We need to experiment with the text to find a useful amount of overlap. Too much overlap repeats information and increases storage requirements.

Some character-based strategies split in the middle of a word to enforce a strict character limit. This is easy to understand, but the broken word may be useless to the embedding model or even suggest the wrong meaning. The example above uses `chunk_overlap=0` and splits at blank lines, so it keeps words intact even when a paragraph exceeds the 110-character target.